# Hierarchical KV pressure-study analysis

This notebook loads experiment JSON files under `experiments/hkv_pressure_study/runs/`,
derives paper-ready tables and figures, and exports them to `tables/` and `plots/`.

All quantities below are computed from the recorded JSON. Timed-out runs are never
treated as completed workloads. Performance-mode validation checks operational
invariants only; it is **not** evidence of output-quality equivalence.
GSM8K resume-quality results are loaded only from `runs/gsm8k_resume_quality/final_n20_batches5_cap512_seq/batch_*` (section 8). Earlier smoke, debug, concurrent-resume, and synchronous-scheduling GSM8K runs are excluded.


## 1. Imports, paths, and plotting style


In [1]:
from __future__ import annotations

import json
import math
import re
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.lines import Line2D
try:
    import seaborn as sns  # optional; not required
except ImportError:
    sns = None


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        marker = candidate / "experiments" / "hkv_pressure_study" / "runs"
        if marker.is_dir():
            return candidate
    raise FileNotFoundError(
        "Could not locate experiments/hkv_pressure_study/runs relative to cwd."
    )


REPO_ROOT = find_repo_root()
STUDY_DIR = REPO_ROOT / "experiments" / "hkv_pressure_study"
RUNS_DIR = STUDY_DIR / "runs"
PLOTS_DIR = STUDY_DIR / "plots"
TABLES_DIR = STUDY_DIR / "tables"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

COLOR_ALL_HOT = "#08306b"  # dark blue
COLOR_MIXED = "#e67e22"  # orange
MODE_COLORS = {"all-hot": COLOR_ALL_HOT, "mixed": COLOR_MIXED}
MODE_LABELS = {"all-hot": "All-HOT", "mixed": "Mixed"}

plt.rcParams.update(
    {
        "figure.dpi": 120,
        "savefig.dpi": 300,
        "savefig.bbox": "tight",
        "savefig.pad_inches": 0.08,
        "font.family": "DejaVu Sans",
        "font.size": 11,
        "axes.titlesize": 12,
        "axes.labelsize": 11,
        "axes.linewidth": 0.8,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": True,
        "grid.alpha": 0.28,
        "grid.linestyle": "--",
        "legend.frameon": False,
        "legend.fontsize": 10,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "axes.prop_cycle": plt.cycler(color=[COLOR_ALL_HOT, COLOR_MIXED, "#4a4a4a"]),
    }
)

SAVED_FIGURES: list[Path] = []


def save_figure(fig: plt.Figure, stem: str) -> list[Path]:
    # Write PDF and PNG under plots/. Overwrites previous exports (idempotent).
    written = []
    for suffix in ("pdf", "png"):
        path = PLOTS_DIR / f"{stem}.{suffix}"
        fig.savefig(path)
        written.append(path)
    SAVED_FIGURES.extend(written)
    return written


def giB(nbytes) -> float | None:
    if nbytes is None or (isinstance(nbytes, float) and math.isnan(nbytes)):
        return None
    return float(nbytes) / (1024**3)


print("repo root:", REPO_ROOT)
print("runs dir:", RUNS_DIR)
print("plots dir:", PLOTS_DIR)
print("tables dir:", TABLES_DIR)
print("seaborn available:", sns is not None)


repo root: /home/shani.dayan/kv_cache_project
runs dir: /home/shani.dayan/kv_cache_project/experiments/hkv_pressure_study/runs
plots dir: /home/shani.dayan/kv_cache_project/experiments/hkv_pressure_study/plots
tables dir: /home/shani.dayan/kv_cache_project/experiments/hkv_pressure_study/tables
seaborn available: False


## 2. Result discovery

JSON files named `all_hot.json` or `mixed.json` are discovered recursively under
`runs/`, excluding `gsm8k_resume_quality/` (loaded separately in section 8).
files are retained in a diagnostics table rather than dropped silently.

Run labels, workload family (`capacity_boundary`, `full_trace_overload`, or
`unclassified`), and repetition IDs are derived from known directory names and
JSON `selection` / `termination` fields. A `final_` prefix is not treated as nominal load. A run is **completed** only when
`termination.timed_out` is false and `termination.status` is `completed`.


In [2]:
REQUIRED_TOP_LEVEL = (
    "kv_mode",
    "termination",
    "validation",
    "hkv_observation",
    "runtime_persistent_kv_memory",
    "selection",
)

BOUNDARY_SESSION_SIZES = {100, 125, 150}


def nested_get(mapping, *keys, default=None):
    cur = mapping
    for key in keys:
        if not isinstance(cur, dict) or key not in cur:
            return default
        cur = cur[key]
    return cur


def percentile(mapping, field: str, name: str):
    block = mapping.get(field)
    if not isinstance(block, dict):
        return None
    return block.get(name)


def parse_repetition(dir_name: str) -> int:
    match = re.search(r"_rep(\d+)$", dir_name)
    if match:
        return int(match.group(1))
    return 1


def parse_workload_family(rel_parent: Path, selected_sessions) -> tuple[str, int | None]:
    """Classify by known run-directory names, not a generic final_ prefix."""
    name = rel_parent.name
    boundary = re.match(r"boundary_(100|125|150)(?:_rep\d+)?$", name)
    if boundary:
        return "capacity_boundary", int(boundary.group(1))
    if name == "final_01_no_prefix_cache":
        size = int(selected_sessions) if selected_sessions is not None else None
        return "full_trace_overload", size
    size = int(selected_sessions) if selected_sessions is not None else None
    return "unclassified", size


def is_completed_run(timed_out, status: str | None) -> bool:
    if timed_out is True:
        return False
    if status is None:
        return False
    return str(status).lower() == "completed"


def extract_row(path: Path, data: dict) -> dict:
    term = data.get("termination") if isinstance(data.get("termination"), dict) else {}
    validation = data.get("validation") if isinstance(data.get("validation"), dict) else {}
    hkv = data.get("hkv_observation") if isinstance(data.get("hkv_observation"), dict) else {}
    runtime = (
        data.get("runtime_persistent_kv_memory")
        if isinstance(data.get("runtime_persistent_kv_memory"), dict)
        else {}
    )
    selection = data.get("selection") if isinstance(data.get("selection"), dict) else {}
    budget = (
        data.get("persistent_kv_memory_budget")
        if isinstance(data.get("persistent_kv_memory_budget"), dict)
        else {}
    )

    rel = path.relative_to(REPO_ROOT)
    rel_parent = path.parent.relative_to(RUNS_DIR)
    kv_mode = data.get("kv_mode")
    timed_out = term.get("timed_out")
    status = term.get("status")
    completed = is_completed_run(timed_out, status)

    selected_sessions = term.get("selected_session_count", selection.get("selected_session_count"))
    selected_requests = term.get("selected_request_count", selection.get("selected_request_count"))
    completed_sessions = term.get("completed_session_count")
    completed_requests = term.get("completed_request_count")

    if selected_requests not in (None, 0) and completed_requests is not None:
        completion_rate = 100.0 * float(completed_requests) / float(selected_requests)
    else:
        completion_rate = None

    family, workload_size = parse_workload_family(rel_parent, selected_sessions)
    if workload_size is None and selected_sessions is not None:
        workload_size = int(selected_sessions)

    hot_bytes = runtime.get("hot_kv_storage_bytes", hkv.get("hot_kv_storage_bytes"))
    warm_bytes = runtime.get("warm_kv_storage_bytes", hkv.get("warm_kv_storage_bytes"))
    map_bytes = runtime.get("hot_to_warm_map_storage_bytes", hkv.get("hot_to_warm_map_storage_bytes"))
    slot_bytes = runtime.get("warm_slot_table_storage_bytes", hkv.get("warm_slot_table_storage_bytes"))
    metadata_bytes = None
    if map_bytes is not None or slot_bytes is not None:
        metadata_bytes = int(map_bytes or 0) + int(slot_bytes or 0)

    total_persistent = runtime.get(
        "actual_persistent_kv_bytes", hkv.get("actual_persistent_kv_bytes")
    )
    configured_budget = runtime.get(
        "configured_total_kv_budget_bytes",
        budget.get("total_kv_budget_bytes"),
    )
    slack = runtime.get("budget_slack_bytes", budget.get("derived_budget_slack_bytes"))

    service_s = term.get("service_window_duration_seconds")
    since_last = term.get("seconds_since_last_completed_turn")
    time_to_last = None
    if service_s is not None and since_last is not None:
        time_to_last = float(service_s) - float(since_last)

    rps = data.get("requests_per_second")
    tps = data.get("output_tokens_per_second_service_window")
    # Timed-out runs must not be treated as complete-workload throughput.
    if not completed:
        rps_complete = None
        tps_complete = None
    else:
        rps_complete = rps
        tps_complete = tps

    errors = validation.get("errors")
    if isinstance(errors, list):
        error_text = "; ".join(str(e) for e in errors)
    else:
        error_text = None if errors is None else str(errors)

    return {
        "source_path": str(rel),
        "run_label": str(rel_parent),
        "run_dir_name": path.parent.name,
        "workload_family": family,
        "workload_size": workload_size,
        "repetition": parse_repetition(path.parent.name),
        "mode": kv_mode,
        "mode_label": MODE_LABELS.get(kv_mode, kv_mode),
        "schema_version": data.get("schema_version"),
        "experiment_mode": data.get("experiment_mode"),
        "selected_sessions": selected_sessions,
        "completed_sessions": completed_sessions,
        "selected_requests": selected_requests,
        "completed_requests": completed_requests,
        "completion_rate_pct": completion_rate,
        "status": status,
        "timed_out": timed_out,
        "completed_run": completed,
        "service_window_duration_s": service_s,
        "seconds_since_last_completed_turn": since_last,
        "time_to_last_completed_turn_s": time_to_last,
        "generated_tokens": term.get("total_generated_tokens", data.get("total_generated_tokens")),
        "requests_per_second": rps,
        "output_tokens_per_second": tps,
        "requests_per_second_completed_workload": rps_complete,
        "output_tokens_per_second_completed_workload": tps_complete,
        "ttft_p50_s": percentile(data, "all_turn_ttft_seconds", "p50"),
        "ttft_p95_s": percentile(data, "all_turn_ttft_seconds", "p95"),
        "resumed_ttft_p50_s": percentile(data, "resumed_turn_ttft_seconds", "p50"),
        "resumed_ttft_p95_s": percentile(data, "resumed_turn_ttft_seconds", "p95"),
        "latency_p50_s": percentile(data, "all_turn_latency_seconds", "p50"),
        "latency_p95_s": percentile(data, "all_turn_latency_seconds", "p95"),
        "resumed_latency_p50_s": percentile(data, "resumed_turn_latency_seconds", "p50"),
        "resumed_latency_p95_s": percentile(data, "resumed_turn_latency_seconds", "p95"),
        "num_gpu_blocks": runtime.get("num_gpu_blocks", hkv.get("num_gpu_blocks")),
        "hot_bytes": hot_bytes,
        "warm_bytes": warm_bytes,
        "metadata_bytes": metadata_bytes,
        "hot_to_warm_map_bytes": map_bytes,
        "warm_slot_table_bytes": slot_bytes,
        "total_persistent_kv_bytes": total_persistent,
        "configured_kv_budget_bytes": configured_budget,
        "budget_slack_bytes": slack,
        "peak_warm_blocks": hkv.get("peak_warm_blocks"),
        "peak_warm_requests": hkv.get("peak_warm_requests"),
        "allocator_consistent": hkv.get("allocator_consistent"),
        "cleanup_complete": hkv.get("cleanup_complete"),
        "validation_passed": validation.get("passed"),
        "validation_errors": error_text,
        "full_validation_performed": validation.get("full_validation_performed"),
        "token_comparison_role": nested_get(data, "baseline_comparison", "token_comparison_role"),
    }


discovered_paths = sorted(
    p
    for p in RUNS_DIR.rglob("*")
    if p.is_file()
    and p.name in {"all_hot.json", "mixed.json"}
    and "gsm8k_resume_quality" not in p.parts
)

rows: list[dict] = []
diagnostics: list[dict] = []

for path in discovered_paths:
    rel = str(path.relative_to(REPO_ROOT))
    record = {
        "source_path": rel,
        "file_name": path.name,
        "issue": None,
        "loaded": False,
    }
    try:
        raw = path.read_text()
    except OSError as exc:
        record["issue"] = f"unreadable: {exc}"
        diagnostics.append(record)
        continue
    if not raw.strip():
        record["issue"] = "empty file"
        diagnostics.append(record)
        continue
    try:
        data = json.loads(raw)
    except json.JSONDecodeError as exc:
        record["issue"] = f"invalid JSON: {exc}"
        diagnostics.append(record)
        continue
    if not isinstance(data, dict):
        record["issue"] = f"top-level JSON is {type(data).__name__}, expected object"
        diagnostics.append(record)
        continue
    missing = [k for k in REQUIRED_TOP_LEVEL if k not in data]
    issues = []
    if missing:
        issues.append("missing keys: " + ", ".join(missing))
    kv_mode = data.get("kv_mode")
    expected_mode = "all-hot" if path.name == "all_hot.json" else "mixed"
    if kv_mode not in {None, expected_mode}:
        issues.append(f"kv_mode={kv_mode!r} incompatible with file name {path.name}")
    if issues:
        record["issue"] = "; ".join(issues)
        record["loaded"] = True
        diagnostics.append(record)
        # Still attempt a partial row so the file is visible, but flag it.
        try:
            row = extract_row(path, data)
            row["parse_warning"] = record["issue"]
            rows.append(row)
        except Exception as exc:  # noqa: BLE001
            record["issue"] += f"; extract failed: {exc}"
        continue
    try:
        row = extract_row(path, data)
        row["parse_warning"] = None
        rows.append(row)
        record["loaded"] = True
        record["issue"] = None
        diagnostics.append(record)
    except Exception as exc:  # noqa: BLE001
        record["issue"] = f"extract failed: {exc}"
        diagnostics.append(record)

diag_df = pd.DataFrame(diagnostics)
summary = pd.DataFrame(rows)

if not summary.empty:
    summary = summary.sort_values(
        ["workload_family", "workload_size", "repetition", "mode"],
        kind="mergesort",
    ).reset_index(drop=True)

print(f"Discovered {len(discovered_paths)} result files under {RUNS_DIR.relative_to(REPO_ROOT)}")
print(f"Parsed rows: {len(summary)}")
skipped = diag_df[diag_df["issue"].notna()] if not diag_df.empty else diag_df
print(f"Files with diagnostics issues: {len(skipped)}")


Discovered 12 result files under experiments/hkv_pressure_study/runs
Parsed rows: 12
Files with diagnostics issues: 0


In [3]:
print("Discovery diagnostics (every discovered file is listed):")
display_cols = ["source_path", "file_name", "loaded", "issue"]
if diag_df.empty:
    print("No files discovered.")
else:
    display(diag_df[display_cols].fillna(""))

ok_files = diag_df[diag_df["issue"].isna()]["source_path"].tolist() if not diag_df.empty else []
issue_files = diag_df[diag_df["issue"].notna()] if not diag_df.empty else diag_df
print("\nLoaded without schema issues:")
for p in ok_files:
    print(" ", p)
if issue_files is not None and not issue_files.empty:
    print("\nSkipped or flagged:")
    for _, r in issue_files.iterrows():
        print(f"  {r['source_path']}: {r['issue']}")
else:
    print("\nNo files skipped.")


Discovery diagnostics (every discovered file is listed):


,source_path,file_name,loaded,issue
0,experiments/hkv_pressure_study/runs/full_text_...,all_hot.json,True,
1,experiments/hkv_pressure_study/runs/full_text_...,mixed.json,True,
2,experiments/hkv_pressure_study/runs/full_text_...,all_hot.json,True,
3,experiments/hkv_pressure_study/runs/full_text_...,mixed.json,True,
4,experiments/hkv_pressure_study/runs/full_text_...,all_hot.json,True,
5,experiments/hkv_pressure_study/runs/full_text_...,mixed.json,True,
6,experiments/hkv_pressure_study/runs/full_text_...,all_hot.json,True,
7,experiments/hkv_pressure_study/runs/full_text_...,mixed.json,True,
8,experiments/hkv_pressure_study/runs/full_text_...,all_hot.json,True,
9,experiments/hkv_pressure_study/runs/full_text_...,mixed.json,True,



Loaded without schema issues:
  experiments/hkv_pressure_study/runs/full_text_8k_cap32/budget_3_5g/boundary_100/all_hot.json
  experiments/hkv_pressure_study/runs/full_text_8k_cap32/budget_3_5g/boundary_100/mixed.json
  experiments/hkv_pressure_study/runs/full_text_8k_cap32/budget_3_5g/boundary_125/all_hot.json
  experiments/hkv_pressure_study/runs/full_text_8k_cap32/budget_3_5g/boundary_125/mixed.json
  experiments/hkv_pressure_study/runs/full_text_8k_cap32/budget_3_5g/boundary_125_rep2/all_hot.json
  experiments/hkv_pressure_study/runs/full_text_8k_cap32/budget_3_5g/boundary_125_rep2/mixed.json
  experiments/hkv_pressure_study/runs/full_text_8k_cap32/budget_3_5g/boundary_125_rep3/all_hot.json
  experiments/hkv_pressure_study/runs/full_text_8k_cap32/budget_3_5g/boundary_125_rep3/mixed.json
  experiments/hkv_pressure_study/runs/full_text_8k_cap32/budget_3_5g/boundary_150/all_hot.json
  experiments/hkv_pressure_study/runs/full_text_8k_cap32/budget_3_5g/boundary_150/mixed.json
  experim

## 3. Validation summary

Each row is one JSON result. `completed_run` is false whenever the workload timed
out, regardless of partial session or request counts. Throughput columns that end
in `_completed_workload` are populated only for completed runs.


In [4]:
VALIDATION_COLUMNS = [
    "run_label",
    "repetition",
    "mode",
    "selected_sessions",
    "completed_sessions",
    "selected_requests",
    "completed_requests",
    "completion_rate_pct",
    "status",
    "timed_out",
    "service_window_duration_s",
    "generated_tokens",
    "requests_per_second",
    "output_tokens_per_second",
    "ttft_p50_s",
    "ttft_p95_s",
    "resumed_ttft_p50_s",
    "resumed_ttft_p95_s",
    "latency_p50_s",
    "latency_p95_s",
    "resumed_latency_p50_s",
    "resumed_latency_p95_s",
    "num_gpu_blocks",
    "hot_bytes",
    "warm_bytes",
    "total_persistent_kv_bytes",
    "peak_warm_blocks",
    "peak_warm_requests",
    "allocator_consistent",
    "cleanup_complete",
    "validation_passed",
]

if summary.empty:
    validation_df = pd.DataFrame(columns=VALIDATION_COLUMNS)
    print("No parsed runs to summarize.")
else:
    validation_df = summary[VALIDATION_COLUMNS].copy()
    validation_df.insert(0, "source_path", summary["source_path"])
    validation_df.insert(1, "completed_run", summary["completed_run"])
    validation_df.insert(2, "workload_family", summary["workload_family"])
    pd.set_option("display.max_columns", None)
    pd.set_option("display.width", 200)
    pd.set_option("display.max_colwidth", 80)
    display(validation_df)

summary_csv = TABLES_DIR / "all_runs_summary.csv"
export_df = summary.copy()
export_df.to_csv(summary_csv, index=False)
print("wrote", summary_csv.relative_to(REPO_ROOT))


,source_path,completed_run,workload_family,run_label,repetition,mode,selected_sessions,completed_sessions,selected_requests,completed_requests,completion_rate_pct,status,timed_out,service_window_duration_s,generated_tokens,requests_per_second,output_tokens_per_second,ttft_p50_s,ttft_p95_s,resumed_ttft_p50_s,resumed_ttft_p95_s,latency_p50_s,latency_p95_s,resumed_latency_p50_s,resumed_latency_p95_s,num_gpu_blocks,hot_bytes,warm_bytes,total_persistent_kv_bytes,peak_warm_blocks,peak_warm_requests,allocator_consistent,cleanup_complete,validation_passed
0,experiments/hkv_pressure_study/runs/full_text_8k_cap32/budget_3_5g/boundary_...,True,capacity_boundary,full_text_8k_cap32/budget_3_5g/boundary_100,1,all-hot,100,100,225,225,100.000000,completed,False,151.392407,3325,1.486204,21.962792,0.481871,9.428911,0.210342,5.056222,1.573963,10.643192,0.433560,5.478599,2048,3758096384,0,3758096384,0,0,True,True,True
1,experiments/hkv_pressure_study/runs/full_text_8k_cap32/budget_3_5g/boundary_...,True,capacity_boundary,full_text_8k_cap32/budget_3_5g/boundary_100,1,mixed,100,100,225,225,100.000000,completed,False,151.722418,3325,1.482971,21.915021,1.140678,15.556574,0.570586,15.618728,4.102788,19.140763,1.342458,17.177967,1517,2783707136,968884224,3757479856,1015,24,True,True,True
2,experiments/hkv_pressure_study/runs/full_text_8k_cap32/budget_3_5g/boundary_...,False,capacity_boundary,full_text_8k_cap32/budget_3_5g/boundary_125,1,all-hot,125,70,275,158,57.454545,timeout,True,600.035168,2328,NaN,NaN,1.631351,10.374402,0.410656,1.354282,4.338401,11.654191,0.651254,4.011224,2048,3758096384,0,3758096384,0,0,True,True,False
3,experiments/hkv_pressure_study/runs/full_text_8k_cap32/budget_3_5g/boundary_...,True,capacity_boundary,full_text_8k_cap32/budget_3_5g/boundary_125,1,mixed,125,125,275,275,100.000000,completed,False,151.712963,4150,1.812634,27.354287,3.021994,24.457299,0.747539,25.992204,6.831810,25.936763,1.509935,26.455998,1517,2783707136,968884224,3757479856,1020,25,True,True,True
4,experiments/hkv_pressure_study/runs/full_text_8k_cap32/budget_3_5g/boundary_...,False,capacity_boundary,full_text_8k_cap32/budget_3_5g/boundary_125_rep2,2,all-hot,125,69,275,154,56.000000,timeout,True,600.022355,2293,NaN,NaN,1.564642,9.767790,0.344733,0.952138,4.211196,11.070235,0.593630,3.593521,2048,3758096384,0,3758096384,0,0,True,True,False
5,experiments/hkv_pressure_study/runs/full_text_8k_cap32/budget_3_5g/boundary_...,True,capacity_boundary,full_text_8k_cap32/budget_3_5g/boundary_125_rep2,2,mixed,125,125,275,275,100.000000,completed,False,151.624799,4150,1.813687,27.370193,2.927633,24.080992,0.748161,25.568773,6.844342,24.825502,1.454717,26.480966,1517,2783707136,968884224,3757479856,1022,27,True,True,True
6,experiments/hkv_pressure_study/runs/full_text_8k_cap32/budget_3_5g/boundary_...,False,capacity_boundary,full_text_8k_cap32/budget_3_5g/boundary_125_rep3,3,all-hot,125,67,275,147,53.454545,timeout,True,600.039299,2224,NaN,NaN,1.710502,9.394662,0.378213,1.171281,4.236020,11.192190,0.600563,3.748798,2048,3758096384,0,3758096384,0,0,True,True,False
7,experiments/hkv_pressure_study/runs/full_text_8k_cap32/budget_3_5g/boundary_...,True,capacity_boundary,full_text_8k_cap32/budget_3_5g/boundary_125_rep3,3,mixed,125,125,275,275,100.000000,completed,False,151.650334,4150,1.813382,27.365584,2.990916,24.237909,0.791991,25.546578,6.736489,25.630018,1.472460,26.430933,1517,2783707136,968884224,3757479856,1023,25,True,True,True
8,experiments/hkv_pressure_study/runs/full_text_8k_cap32/budget_3_5g/boundary_...,False,capacity_boundary,full_text_8k_cap32/budget_3_5g/boundary_150,1,all-hot,150,70,330,158,47.878788,timeout,True,600.037410,2328,NaN,NaN,1.680260,10.541904,0.423276,1.305928,4.453368,11.834453,0.584945,3.842092,2048,3758096384,0,3758096384,0,0,True,True,False
9,experiments/hkv_pressure_study/runs/full_text_8k_cap32/budget_3_5g/boundary_...,False,capacity_boundary,full_text_8k_cap32/budget_3_5g/boundary_150,1,mixed,150,85,330,190,57.575758,timeout,True,600.035002

wrote experiments/hkv_pressure_study/tables/all_runs_summary.csv


## 4. Boundary experiment: 100 / 125 / 150 sessions

**Figure.** Completed-request percentage versus selected session count for All-HOT
and Mixed under the matched persistent-KV budget. Filled markers are completed
workloads; open markers are timed-out (partial) runs. Where repetitions exist,
the series shows the mean and the min–max range.


In [5]:
boundary = summary[
    (summary["workload_family"] == "capacity_boundary")
    & (summary["workload_size"].isin(BOUNDARY_SESSION_SIZES))
].copy() if not summary.empty else summary

if boundary.empty:
    print("No boundary runs with 100, 125, or 150 selected sessions were found.")
else:
    print(f"Boundary rows: {len(boundary)}")
    display(
        boundary[
            [
                "run_label",
                "repetition",
                "mode_label",
                "selected_sessions",
                "completed_sessions",
                "completed_requests",
                "selected_requests",
                "completion_rate_pct",
                "timed_out",
                "completed_run",
            ]
        ]
    )

    fig, ax = plt.subplots(figsize=(6.4, 4.2))
    x_ticks = sorted(boundary["selected_sessions"].dropna().unique())

    for mode, color in MODE_COLORS.items():
        sub = boundary[boundary["mode"] == mode]
        if sub.empty:
            continue
        # Individual repetition points.
        completed_pts = sub[sub["completed_run"]]
        timeout_pts = sub[~sub["completed_run"]]
        if not completed_pts.empty:
            ax.scatter(
                completed_pts["selected_sessions"],
                completed_pts["completion_rate_pct"],
                color=color,
                s=46,
                zorder=4,
                marker="o",
                edgecolors="white",
                linewidths=0.6,
                label=None,
            )
        if not timeout_pts.empty:
            ax.scatter(
                timeout_pts["selected_sessions"],
                timeout_pts["completion_rate_pct"],
                facecolors="none",
                edgecolors=color,
                s=54,
                zorder=4,
                marker="o",
                linewidths=1.4,
                label=None,
            )

        grouped = sub.groupby("selected_sessions")["completion_rate_pct"]
        means = grouped.mean()
        mins = grouped.min()
        maxs = grouped.max()
        counts = grouped.count()
        xs = means.index.to_numpy(dtype=float)
        ys = means.to_numpy(dtype=float)
        yerr = np.vstack([ys - mins.to_numpy(dtype=float), maxs.to_numpy(dtype=float) - ys])
        yerr = np.clip(yerr, 0, None)
        ax.errorbar(
            xs,
            ys,
            yerr=yerr,
            color=color,
            linewidth=1.6,
            capsize=3.5,
            capthick=1.2,
            zorder=3,
            label=MODE_LABELS[mode],
            marker="s",
            markersize=5,
        )

    ax.set_xticks(x_ticks)
    ax.set_xlabel("Selected sessions")
    ax.set_ylabel("Completed requests (%)")
    ax.set_ylim(-3, 108)
    ax.set_title("Capacity-pressure completion vs. session count")
    legend_elems = [
        Line2D([0], [0], color=COLOR_ALL_HOT, marker="s", linewidth=1.6, label="All-HOT"),
        Line2D([0], [0], color=COLOR_MIXED, marker="s", linewidth=1.6, label="Mixed"),
        Line2D(
            [0],
            [0],
            marker="o",
            color="0.3",
            markerfacecolor="0.3",
            linestyle="None",
            label="Completed run",
        ),
        Line2D(
            [0],
            [0],
            marker="o",
            color="0.3",
            markerfacecolor="none",
            linestyle="None",
            markeredgewidth=1.4,
            label="Timed-out run",
        ),
    ]
    ax.legend(handles=legend_elems, loc="lower left")
    save_figure(fig, "boundary_completion_rate")
    plt.show()


Boundary rows: 10


,run_label,repetition,mode_label,selected_sessions,completed_sessions,completed_requests,selected_requests,completion_rate_pct,timed_out,completed_run
0,full_text_8k_cap32/budget_3_5g/boundary_100,1,All-HOT,100,100,225,225,100.000000,False,True
1,full_text_8k_cap32/budget_3_5g/boundary_100,1,Mixed,100,100,225,225,100.000000,False,True
2,full_text_8k_cap32/budget_3_5g/boundary_125,1,All-HOT,125,70,158,275,57.454545,True,False
3,full_text_8k_cap32/budget_3_5g/boundary_125,1,Mixed,125,125,275,275,100.000000,False,True
4,full_text_8k_cap32/budget_3_5g/boundary_125_rep2,2,All-HOT,125,69,154,275,56.000000,True,False
5,full_text_8k_cap32/budget_3_5g/boundary_125_rep2,2,Mixed,125,125,275,275,100.000000,False,True
6,full_text_8k_cap32/budget_3_5g/boundary_125_rep3,3,All-HOT,125,67,147,275,53.454545,True,False
7,full_text_8k_cap32/budget_3_5g/boundary_125_rep3,3,Mixed,125,125,275,275,100.000000,False,True
8,full_text_8k_cap32/budget_3_5g/boundary_150,1,All-HOT,150,70,158,330,47.878788,True,False
9,full_text_8k_cap32/budget_3_5g/boundary_150,1,Mixed,150,85,190,330,57.575758,True,False


## 5. Repeated 125-session experiment

The 125-session workload is the reported operating point with repeated trials.
Completed-request and completed-session counts are shown for every repetition,
including timed-out All-HOT runs (partial progress). Throughput and latency
averages below use **only completed Mixed workloads**; timed-out All-HOT runs
are excluded from those aggregates.

Time to last completed turn is
`service_window_duration_seconds − seconds_since_last_completed_turn`.
For a completed run this equals the service-window duration.


In [6]:
rep125 = (
    boundary[boundary["workload_size"] == 125].copy()
    if not boundary.empty
    else pd.DataFrame()
)

if rep125.empty:
    print("No 125-session boundary repetitions were found.")
    rep125_table = pd.DataFrame()
    agg125 = pd.DataFrame()
else:
    show_cols = [
        "run_label",
        "repetition",
        "mode_label",
        "completed_run",
        "timed_out",
        "selected_sessions",
        "completed_sessions",
        "selected_requests",
        "completed_requests",
        "completion_rate_pct",
        "generated_tokens",
        "service_window_duration_s",
        "seconds_since_last_completed_turn",
        "time_to_last_completed_turn_s",
        "requests_per_second_completed_workload",
        "output_tokens_per_second_completed_workload",
        "ttft_p50_s",
        "ttft_p95_s",
        "resumed_ttft_p50_s",
        "resumed_ttft_p95_s",
        "latency_p50_s",
        "latency_p95_s",
        "peak_warm_blocks",
        "peak_warm_requests",
        "validation_passed",
    ]
    rep125_table = (
        rep125.sort_values(["repetition", "mode"], kind="mergesort")[show_cols]
    )
    display(rep125_table)

    def _agg(series: pd.Series) -> pd.Series:
        s = pd.to_numeric(series, errors="coerce").dropna()
        if s.empty:
            return pd.Series({"n": 0, "mean": np.nan, "std": np.nan, "min": np.nan, "max": np.nan})
        return pd.Series(
            {
                "n": int(s.count()),
                "mean": float(s.mean()),
                "std": float(s.std(ddof=1)) if len(s) > 1 else 0.0,
                "min": float(s.min()),
                "max": float(s.max()),
            }
        )

    metric_specs = [
        ("completed_sessions", False),
        ("completed_requests", False),
        ("completion_rate_pct", False),
        ("time_to_last_completed_turn_s", False),
        ("generated_tokens", False),
        ("service_window_duration_s", False),
        ("requests_per_second_completed_workload", True),
        ("output_tokens_per_second_completed_workload", True),
        ("ttft_p50_s", True),
        ("ttft_p95_s", True),
        ("resumed_ttft_p50_s", True),
        ("resumed_ttft_p95_s", True),
        ("latency_p50_s", True),
        ("latency_p95_s", True),
        ("resumed_latency_p50_s", True),
        ("resumed_latency_p95_s", True),
    ]

    agg_rows = []
    for mode, grp in rep125.groupby("mode"):
        for metric, completed_only in metric_specs:
            src = grp[grp["completed_run"]] if completed_only else grp
            stats = _agg(src[metric]) if metric in src.columns else _agg(pd.Series(dtype=float))
            agg_rows.append(
                {
                    "mode": mode,
                    "mode_label": MODE_LABELS.get(mode, mode),
                    "metric": metric,
                    "completed_workloads_only": completed_only,
                    **stats.to_dict(),
                    "timed_out_repetitions_excluded": int((~grp["completed_run"]).sum())
                    if completed_only
                    else 0,
                }
            )
    agg125 = pd.DataFrame(agg_rows)
    print("\nAggregate statistics for 125-session repetitions")
    print("(throughput/latency rows use completed workloads only):")
    display(agg125)

    rep125_csv = TABLES_DIR / "boundary_125_repetitions.csv"
    agg125_csv = TABLES_DIR / "boundary_125_aggregate.csv"
    rep125_table.to_csv(rep125_csv, index=False)
    agg125.to_csv(agg125_csv, index=False)
    print("wrote", rep125_csv.relative_to(REPO_ROOT))
    print("wrote", agg125_csv.relative_to(REPO_ROOT))


,run_label,repetition,mode_label,completed_run,timed_out,selected_sessions,completed_sessions,selected_requests,completed_requests,completion_rate_pct,generated_tokens,service_window_duration_s,seconds_since_last_completed_turn,time_to_last_completed_turn_s,requests_per_second_completed_workload,output_tokens_per_second_completed_workload,ttft_p50_s,ttft_p95_s,resumed_ttft_p50_s,resumed_ttft_p95_s,latency_p50_s,latency_p95_s,peak_warm_blocks,peak_warm_requests,validation_passed
2,full_text_8k_cap32/budget_3_5g/boundary_125,1,All-HOT,False,True,125,70,275,158,57.454545,2328,600.035168,570.802381,29.232787,NaN,NaN,1.631351,10.374402,0.410656,1.354282,4.338401,11.654191,0,0,False
3,full_text_8k_cap32/budget_3_5g/boundary_125,1,Mixed,True,False,125,125,275,275,100.000000,4150,151.712963,0.000000,151.712963,1.812634,27.354287,3.021994,24.457299,0.747539,25.992204,6.831810,25.936763,1020,25,True
4,full_text_8k_cap32/budget_3_5g/boundary_125_rep2,2,All-HOT,False,True,125,69,275,154,56.000000,2293,600.022355,578.044192,21.978163,NaN,NaN,1.564642,9.767790,0.344733,0.952138,4.211196,11.070235,0,0,False
5,full_text_8k_cap32/budget_3_5g/boundary_125_rep2,2,Mixed,True,False,125,125,275,275,100.000000,4150,151.624799,0.000000,151.624799,1.813687,27.370193,2.927633,24.080992,0.748161,25.568773,6.844342,24.825502,1022,27,True
6,full_text_8k_cap32/budget_3_5g/boundary_125_rep3,3,All-HOT,False,True,125,67,275,147,53.454545,2224,600.039299,570.797352,29.241948,NaN,NaN,1.710502,9.394662,0.378213,1.171281,4.236020,11.192190,0,0,False
7,full_text_8k_cap32/budget_3_5g/boundary_125_rep3,3,Mixed,True,False,125,125,275,275,100.000000,4150,151.650334,0.000000,151.650334,1.813382,27.365584,2.990916,24.237909,0.791991,25.546578,6.736489,25.630018,1023,25,True



Aggregate statistics for 125-session repetitions
(throughput/latency rows use completed workloads only):


,mode,mode_label,metric,completed_workloads_only,n,mean,std,min,max,timed_out_repetitions_excluded
0,all-hot,All-HOT,completed_sessions,False,3.0,68.666667,1.527525,67.000000,70.000000,0
1,all-hot,All-HOT,completed_requests,False,3.0,153.000000,5.567764,147.000000,158.000000,0
2,all-hot,All-HOT,completion_rate_pct,False,3.0,55.636364,2.024642,53.454545,57.454545,0
3,all-hot,All-HOT,time_to_last_completed_turn_s,False,3.0,26.817633,4.191106,21.978163,29.241948,0
4,all-hot,All-HOT,generated_tokens,False,3.0,2281.666667,52.918176,2224.000000,2328.000000,0
5,all-hot,All-HOT,service_window_duration_s,False,3.0,600.032274,0.008835,600.022355,600.039299,0
6,all-hot,All-HOT,requests_per_second_completed_workload,True,0.0,NaN,NaN,NaN,NaN,3
7,all-hot,All-HOT,output_tokens_per_second_completed_workload,True,0.0,NaN,NaN,NaN,NaN,3
8,all-hot,All-HOT,ttft_p50_s,True,0.0,NaN,NaN,NaN,NaN,3
9,all-hot,All-HOT,ttft_p95_s,True,0.0,NaN,NaN,NaN,NaN,3


wrote experiments/hkv_pressure_study/tables/boundary_125_repetitions.csv
wrote experiments/hkv_pressure_study/tables/boundary_125_aggregate.csv


In [7]:
def paired_dotplot(df: pd.DataFrame, value_col: str, ylabel: str, stem: str, title: str) -> None:
    if df.empty:
        print(f"Skipping {stem}: no 125-session rows.")
        return
    fig, ax = plt.subplots(figsize=(4.6, 4.0))
    x_pos = {"all-hot": 0, "mixed": 1}
    for rep, sub in df.groupby("repetition"):
        xs, ys, colors, completed_flags = [], [], [], []
        for _, row in sub.sort_values("mode").iterrows():
            if row["mode"] not in x_pos:
                continue
            xs.append(x_pos[row["mode"]])
            ys.append(row[value_col])
            colors.append(MODE_COLORS[row["mode"]])
            completed_flags.append(bool(row["completed_run"]))
        if len(xs) == 2:
            ax.plot(xs, ys, color="0.65", linewidth=1.0, zorder=2)
        for x, y, c, done in zip(xs, ys, colors, completed_flags):
            ax.scatter(
                [x],
                [y],
                s=70,
                color=c if done else "none",
                edgecolors=c,
                linewidths=1.5,
                zorder=3,
            )
            ax.annotate(
                f"r{int(rep)}",
                (x, y),
                textcoords="offset points",
                xytext=(6, 4),
                fontsize=8,
                color="0.25",
            )
    ax.set_xticks([0, 1])
    ax.set_xticklabels(["All-HOT", "Mixed"])
    ax.set_xlim(-0.45, 1.45)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    legend_elems = [
        Line2D(
            [0], [0], marker="o", color="0.3", markerfacecolor="0.3",
            linestyle="None", label="Completed",
        ),
        Line2D(
            [0], [0], marker="o", color="0.3", markerfacecolor="none",
            linestyle="None", markeredgewidth=1.4, label="Timed out",
        ),
    ]
    ax.legend(handles=legend_elems, loc="best")
    save_figure(fig, stem)
    plt.show()


if not rep125.empty:
    paired_dotplot(
        rep125,
        "completed_requests",
        "Completed requests",
        "boundary_125_completed_requests",
        "125-session repetitions: completed requests",
    )
    paired_dotplot(
        rep125,
        "completed_sessions",
        "Completed sessions",
        "boundary_125_completed_sessions",
        "125-session repetitions: completed sessions",
    )


In [8]:
if rep125.empty:
    print("No 125-session rows for time-to-last-completed-turn.")
else:
    display(
        rep125[
            [
                "repetition",
                "mode_label",
                "completed_run",
                "service_window_duration_s",
                "seconds_since_last_completed_turn",
                "time_to_last_completed_turn_s",
            ]
        ].sort_values(["repetition", "mode_label"])
    )

    fig, ax = plt.subplots(figsize=(4.6, 4.0))
    x_pos = {"all-hot": 0, "mixed": 1}
    for rep, sub in rep125.groupby("repetition"):
        xs, ys = [], []
        for _, row in sub.sort_values("mode").iterrows():
            if row["mode"] not in x_pos:
                continue
            x = x_pos[row["mode"]]
            y = row["time_to_last_completed_turn_s"]
            xs.append(x)
            ys.append(y)
            done = bool(row["completed_run"])
            c = MODE_COLORS[row["mode"]]
            ax.scatter(
                [x], [y], s=70, color=c if done else "none",
                edgecolors=c, linewidths=1.5, zorder=3,
            )
            ax.annotate(
                f"r{int(rep)}", (x, y), textcoords="offset points",
                xytext=(6, 4), fontsize=8, color="0.25",
            )
        if len(xs) == 2:
            ax.plot(xs, ys, color="0.65", linewidth=1.0, zorder=2)
    ax.set_xticks([0, 1])
    ax.set_xticklabels(["All-HOT", "Mixed"])
    ax.set_xlim(-0.45, 1.45)
    ax.set_ylabel("Time to last completed turn (s)")
    ax.set_title("125-session repetitions: time to last completed turn")
    save_figure(fig, "boundary_125_time_to_last_completed_turn")
    plt.show()


,repetition,mode_label,completed_run,service_window_duration_s,seconds_since_last_completed_turn,time_to_last_completed_turn_s
2,1,All-HOT,False,600.035168,570.802381,29.232787
3,1,Mixed,True,151.712963,0.000000,151.712963
4,2,All-HOT,False,600.022355,578.044192,21.978163
5,2,Mixed,True,151.624799,0.000000,151.624799
6,3,All-HOT,False,600.039299,570.797352,29.241948
7,3,Mixed,True,151.650334,0.000000,151.650334


## 6. Persistent-memory composition

All-HOT and Mixed share a configured persistent-KV byte budget. Mixed spends part
of that budget on WARM storage plus persistent metadata (HOT↔WARM map and WARM
slot table). The stacked bars use runtime-recorded byte counts. Budget match is
checked as
`configured_total_kv_budget_bytes − actual_persistent_kv_bytes`
versus the recorded `budget_slack_bytes`.


In [9]:
if summary.empty:
    memory_df = pd.DataFrame()
    print("No runs available for memory composition.")
else:
    memory_df = summary[
        [
            "run_label",
            "repetition",
            "mode",
            "mode_label",
            "workload_family",
            "workload_size",
            "completed_run",
            "num_gpu_blocks",
            "hot_bytes",
            "warm_bytes",
            "hot_to_warm_map_bytes",
            "warm_slot_table_bytes",
            "metadata_bytes",
            "total_persistent_kv_bytes",
            "configured_kv_budget_bytes",
            "budget_slack_bytes",
        ]
    ].copy()

    computed_sum = (
        memory_df["hot_bytes"].fillna(0)
        + memory_df["warm_bytes"].fillna(0)
        + memory_df["metadata_bytes"].fillna(0)
    )
    memory_df["hot_warm_metadata_sum_bytes"] = computed_sum
    memory_df["sum_matches_total"] = (
        memory_df["total_persistent_kv_bytes"].notna()
        & (computed_sum == memory_df["total_persistent_kv_bytes"].fillna(-1))
    )
    slack_computed = (
        memory_df["configured_kv_budget_bytes"] - memory_df["total_persistent_kv_bytes"]
    )
    memory_df["computed_slack_bytes"] = slack_computed
    memory_df["within_recorded_slack"] = (
        memory_df["budget_slack_bytes"].notna()
        & memory_df["computed_slack_bytes"].notna()
        & (memory_df["computed_slack_bytes"] == memory_df["budget_slack_bytes"])
        & (memory_df["computed_slack_bytes"] >= 0)
    )
    memory_df["hot_GiB"] = memory_df["hot_bytes"].map(giB)
    memory_df["warm_GiB"] = memory_df["warm_bytes"].map(giB)
    memory_df["metadata_GiB"] = memory_df["metadata_bytes"].map(giB)
    memory_df["total_GiB"] = memory_df["total_persistent_kv_bytes"].map(giB)
    memory_df["budget_GiB"] = memory_df["configured_kv_budget_bytes"].map(giB)

    mem_csv = TABLES_DIR / "memory_composition.csv"
    memory_df.to_csv(mem_csv, index=False)
    print("wrote", mem_csv.relative_to(REPO_ROOT))
    display(
        memory_df[
            [
                "run_label",
                "mode_label",
                "hot_bytes",
                "warm_bytes",
                "metadata_bytes",
                "total_persistent_kv_bytes",
                "configured_kv_budget_bytes",
                "budget_slack_bytes",
                "within_recorded_slack",
                "sum_matches_total",
            ]
        ]
    )

    n_ok = int(memory_df["within_recorded_slack"].sum())
    n_all = len(memory_df)
    print(
        f"Budget slack match: {n_ok}/{n_all} runs have "
        "configured − actual == recorded budget_slack_bytes (≥ 0)."
    )

    # Representative composition per mode: unique byte layouts across parsed runs.
    layout_cols = ["mode", "hot_bytes", "warm_bytes", "metadata_bytes", "total_persistent_kv_bytes"]
    layouts = memory_df.dropna(subset=["hot_bytes"]).drop_duplicates(layout_cols)

    fig, ax = plt.subplots(figsize=(6.6, 4.3))
    # Four conceptual stacks requested: All-HOT HOT; Mixed HOT; Mixed WARM; Mixed metadata.
    labels = []
    bottoms = []
    hot_vals = []
    warm_vals = []
    meta_vals = []
    for mode in ("all-hot", "mixed"):
        sub = layouts[layouts["mode"] == mode]
        if sub.empty:
            continue
        # If layouts differ, plot each unique layout.
        for i, row in sub.reset_index(drop=True).iterrows():
            suffix = "" if len(sub) == 1 else f" #{i+1}"
            labels.append(f"{MODE_LABELS[mode]}{suffix}")
            hot_vals.append(giB(row["hot_bytes"]) or 0.0)
            warm_vals.append(giB(row["warm_bytes"]) or 0.0)
            meta_vals.append(giB(row["metadata_bytes"]) or 0.0)

    x = np.arange(len(labels))
    width = 0.62
    ax.bar(x, hot_vals, width, label="HOT storage", color=COLOR_ALL_HOT)
    ax.bar(x, warm_vals, width, bottom=hot_vals, label="WARM storage", color=COLOR_MIXED)
    bottom_meta = np.array(hot_vals) + np.array(warm_vals)
    ax.bar(
        x,
        meta_vals,
        width,
        bottom=bottom_meta,
        label="Persistent metadata",
        color="#bdbdbd",
    )
    ax.set_xticks(x)
    ax.set_xticklabels(labels)
    ax.set_ylabel("Persistent KV storage (GiB)")
    ax.set_title("Persistent-memory composition under a matched byte budget")
    ax.legend(loc="upper right")
    if memory_df["configured_kv_budget_bytes"].notna().any():
        budget_gib = giB(memory_df["configured_kv_budget_bytes"].dropna().iloc[0])
        ax.axhline(budget_gib, color="0.35", linestyle=":", linewidth=1.0, label=None)
        ax.text(
            0.01,
            budget_gib,
            f"  configured budget {budget_gib:.3f} GiB",
            va="bottom",
            ha="left",
            fontsize=8,
            color="0.35",
            transform=ax.get_yaxis_transform(),
        )
    save_figure(fig, "memory_composition_stacked")
    plt.show()


wrote experiments/hkv_pressure_study/tables/memory_composition.csv


,run_label,mode_label,hot_bytes,warm_bytes,metadata_bytes,total_persistent_kv_bytes,configured_kv_budget_bytes,budget_slack_bytes,within_recorded_slack,sum_matches_total
0,full_text_8k_cap32/budget_3_5g/boundary_100,All-HOT,3758096384,0,0,3758096384,3758096384,0,True,True
1,full_text_8k_cap32/budget_3_5g/boundary_100,Mixed,2783707136,968884224,4888496,3757479856,3758096384,616528,True,True
2,full_text_8k_cap32/budget_3_5g/boundary_125,All-HOT,3758096384,0,0,3758096384,3758096384,0,True,True
3,full_text_8k_cap32/budget_3_5g/boundary_125,Mixed,2783707136,968884224,4888496,3757479856,3758096384,616528,True,True
4,full_text_8k_cap32/budget_3_5g/boundary_125_rep2,All-HOT,3758096384,0,0,3758096384,3758096384,0,True,True
5,full_text_8k_cap32/budget_3_5g/boundary_125_rep2,Mixed,2783707136,968884224,4888496,3757479856,3758096384,616528,True,True
6,full_text_8k_cap32/budget_3_5g/boundary_125_rep3,All-HOT,3758096384,0,0,3758096384,3758096384,0,True,True
7,full_text_8k_cap32/budget_3_5g/boundary_125_rep3,Mixed,2783707136,968884224,4888496,3757479856,3758096384,616528,True,True
8,full_text_8k_cap32/budget_3_5g/boundary_150,All-HOT,3758096384,0,0,3758096384,3758096384,0,True,True
9,full_text_8k_cap32/budget_3_5g/boundary_150,Mixed,2783707136,968884224,4888496,3757479856,3758096384,616528,True,True


Budget slack match: 12/12 runs have configured − actual == recorded budget_slack_bytes (≥ 0).


## 7. Full-trace overload and nominal-load availability

`final_01_no_prefix_cache` is a **full-trace overload** experiment under the same
3.5 GiB pressure configuration (16,328 selected sessions, 30,830 selected requests,
max input length 8,176, max model length 9,216, max 32 generated tokens on final
turns). Both All-HOT and Mixed timed out. It is **not** a nominal-load run.

Directory names starting with `final_` are not treated as nominal load.
Nominal-load results are unavailable in the currently discovered `runs/` directory.


In [10]:
overload = (
    summary[summary["workload_family"] == "full_trace_overload"].copy()
    if not summary.empty
    else pd.DataFrame()
)

if overload.empty:
    print("No full-trace overload result files were found.")
    overload_table = pd.DataFrame()
else:
    overload_table = overload.copy()
    if (
        overload_table["selected_sessions"].notna().any()
        and overload_table["completed_sessions"].notna().any()
    ):
        overload_table["session_completion_pct"] = (
            100.0
            * overload_table["completed_sessions"].astype(float)
            / overload_table["selected_sessions"].astype(float)
        )
    else:
        overload_table["session_completion_pct"] = None
    overload_table["request_completion_pct"] = overload_table["completion_rate_pct"]

    overload_cols = [
        "run_label",
        "mode_label",
        "timed_out",
        "status",
        "selected_sessions",
        "completed_sessions",
        "session_completion_pct",
        "selected_requests",
        "completed_requests",
        "request_completion_pct",
        "generated_tokens",
        "service_window_duration_s",
        "seconds_since_last_completed_turn",
        "time_to_last_completed_turn_s",
        "peak_warm_blocks",
        "peak_warm_requests",
        "total_persistent_kv_bytes",
        "allocator_consistent",
        "cleanup_complete",
    ]
    overload_table = overload_table[overload_cols].sort_values("mode_label")
    overload_table = overload_table.rename(
        columns={"time_to_last_completed_turn_s": "time_of_last_completed_turn_s"}
    )
    print("Full-trace overload (same 3.5 GiB pressure configuration):")
    display(overload_table)

    overload_csv = TABLES_DIR / "full_trace_overload.csv"
    overload_table.to_csv(overload_csv, index=False)
    print("wrote", overload_csv.relative_to(REPO_ROOT))

nominal = (
    summary[summary["workload_family"] == "nominal_load"].copy()
    if not summary.empty
    else pd.DataFrame()
)

print()
if nominal.empty:
    print(
        "Nominal-load data are unavailable in the currently discovered run directory "
        "experiments/hkv_pressure_study/runs/. "
        "No result files are classified as nominal_load."
    )
else:
    completed_nominal = nominal[nominal["completed_run"]]
    if completed_nominal.empty:
        print(
            "Nominal-load files were found but none completed; "
            "throughput and latency are not summarized as complete-workload results."
        )
        display(nominal)
    else:
        latency_cols = [
            "run_label",
            "mode_label",
            "repetition",
            "requests_per_second_completed_workload",
            "output_tokens_per_second_completed_workload",
            "ttft_p50_s",
            "ttft_p95_s",
            "resumed_ttft_p50_s",
            "resumed_ttft_p95_s",
            "latency_p50_s",
            "latency_p95_s",
            "resumed_latency_p50_s",
            "resumed_latency_p95_s",
        ]
        print("Completed nominal-load throughput and latency:")
        display(completed_nominal[latency_cols])


Full-trace overload (same 3.5 GiB pressure configuration):


,run_label,mode_label,timed_out,status,selected_sessions,completed_sessions,session_completion_pct,selected_requests,completed_requests,request_completion_pct,generated_tokens,service_window_duration_s,seconds_since_last_completed_turn,time_of_last_completed_turn_s,peak_warm_blocks,peak_warm_requests,total_persistent_kv_bytes,allocator_consistent,cleanup_complete
10,full_text_8k_cap32/budget_3_5g/final_01_no_prefix_cache,All-HOT,True,timeout,16328,60,0.367467,30830,124,0.402206,1984,604.195840,574.961265,29.234575,0,0,3758096384,True,True
11,full_text_8k_cap32/budget_3_5g/final_01_no_prefix_cache,Mixed,True,timeout,16328,91,0.557325,30830,177,0.574116,2998,604.037627,453.372929,150.664698,989,16,3757479856,True,True


wrote experiments/hkv_pressure_study/tables/full_trace_overload.csv

Nominal-load data are unavailable in the currently discovered run directory experiments/hkv_pressure_study/runs/. No result files are classified as nominal_load.


## 8. GSM8K resume-quality (sequential turn-2)

This section uses **only** the four `batch_*` directories under

`experiments/hkv_pressure_study/runs/gsm8k_resume_quality/final_n20_batches5_cap512_seq`.

It does **not** include earlier `max_tokens=64` runs, smoke or debug runs,
failed asynchronous concurrent-resume runs, or failed synchronous-scheduling runs.

All-HOT and Mixed share the same questions, prompts, and generation policy.
Turn 2 was resumed sequentially with `async_scheduling=true`. Mixed requires
complete-history WARM residency before resume and a mixed-read step during turn 2.
Exact-match rates are descriptive counts on **n = 20** questions. A five
percentage-point difference is **not** evidence that WARM quantization improves accuracy.


In [11]:
GSM8K_ROOT = (
    RUNS_DIR
    / "gsm8k_resume_quality"
    / "final_n20_batches5_cap512_seq"
)
GSM8K_BATCH_DIRS = sorted(
    p for p in GSM8K_ROOT.iterdir() if p.is_dir() and p.name.startswith("batch_")
)

gsm8k_files = []
for batch_dir in GSM8K_BATCH_DIRS:
    for name in ("all_hot.json", "mixed.json"):
        path = batch_dir / name
        if not path.is_file():
            raise FileNotFoundError(path)
        gsm8k_files.append(path)

print("GSM8K root:", GSM8K_ROOT.relative_to(REPO_ROOT))
print("batch directories:", [p.name for p in GSM8K_BATCH_DIRS])
print("JSON files:")
for path in gsm8k_files:
    print(" ", path.relative_to(REPO_ROOT))

assert len(GSM8K_BATCH_DIRS) == 4, GSM8K_BATCH_DIRS
assert len(gsm8k_files) == 8, len(gsm8k_files)
n_all_hot = sum(p.name == "all_hot.json" for p in gsm8k_files)
n_mixed = sum(p.name == "mixed.json" for p in gsm8k_files)
print(f"All-HOT JSON files: {n_all_hot}")
print(f"Mixed JSON files: {n_mixed}")
assert n_all_hot == 4 and n_mixed == 4

file_rows = []
session_rows = []

for path in gsm8k_files:
    data = json.loads(path.read_text(encoding="utf-8"))
    engine = data.get("engine_configuration") or {}
    validation = data.get("validation") or {}
    hkv = data.get("hkv_observation") or {}
    quality = data.get("quality") or {}
    kv_mode = data["kv_mode"]
    async_scheduling = bool(
        data.get("async_scheduling", engine.get("async_scheduling"))
    )
    sequential_resume = bool(
        data.get("sequential_resume", engine.get("sequential_resume"))
    )
    val_passed = bool(validation.get("passed"))
    n_eligible = int(quality.get("n_eligible") or 0)
    n_selected = int(quality.get("n_selected") or 0)
    warm_observed = bool(hkv.get("warm_observed"))
    allocator_ok = bool(hkv.get("allocator_consistent"))
    cleanup_ok = bool(hkv.get("cleanup_complete"))
    file_rows.append(
        {
            "batch": path.parent.name,
            "mode": kv_mode,
            "source_path": str(path.relative_to(REPO_ROOT)),
            "async_scheduling": async_scheduling,
            "sequential_resume": sequential_resume,
            "validation_passed": val_passed,
            "n_selected": n_selected,
            "n_eligible": n_eligible,
            "n_exact_match": int(quality.get("n_exact_match") or 0),
            "n_parse_failures": int(quality.get("n_parse_failures") or 0),
            "warm_observed": warm_observed,
            "peak_warm_blocks": hkv.get("peak_warm_blocks"),
            "allocator_consistent": allocator_ok,
            "cleanup_complete": cleanup_ok,
        }
    )
    for sess in data.get("sessions") or []:
        token_ids = sess.get("generated_token_ids") or []
        session_rows.append(
            {
                "batch": path.parent.name,
                "mode": kv_mode,
                "session_id": sess.get("session_id"),
                "question_index": sess.get("question_index"),
                "gold_value": sess.get("gold_value"),
                "extracted_value": sess.get("extracted_value"),
                "exact_match": bool(sess.get("exact_match")),
                "parse_failure": bool(sess.get("parse_failure")),
                "final_text": sess.get("final_text") or "",
                "generated_token_count": len(token_ids),
                "mixed_read_steps": int(sess.get("mixed_read_steps") or 0),
                "warm_logical_blocks_observed": int(
                    sess.get("warm_logical_blocks_observed") or 0
                ),
                "pre_resume_warm_confirmed": bool(
                    sess.get("pre_resume_warm_confirmed")
                ),
                "eligible": sess.get("exclusion_reason") is None,
                "exclusion_reason": sess.get("exclusion_reason"),
                "completed_turns": sess.get("completed_turns"),
                "error": sess.get("error"),
            }
        )

file_df = pd.DataFrame(file_rows).sort_values(["batch", "mode"]).reset_index(drop=True)
sessions = (
    pd.DataFrame(session_rows).sort_values(["question_index", "mode"]).reset_index(drop=True)
)

print("\n=== File-level verification ===")
display(file_df)

assert file_df["async_scheduling"].all()
assert file_df["sequential_resume"].all()
assert file_df["validation_passed"].all()
mixed_files = file_df[file_df["mode"] == "mixed"]
assert int(mixed_files["n_eligible"].sum()) == 20
assert mixed_files["warm_observed"].all()
assert file_df["allocator_consistent"].all()
assert file_df["cleanup_complete"].all()
print("async_scheduling=true for every file")
print("sequential_resume=true for every file")
print("validation passed for every file")
print("all 20 Mixed questions eligible")
print("WARM observed in every Mixed batch")
print("allocator consistency and cleanup passed everywhere")

hot = sessions[sessions["mode"] == "all-hot"].set_index("question_index")
mix = sessions[sessions["mode"] == "mixed"].set_index("question_index")
assert list(hot.index) == list(mix.index)
assert len(hot) == 20

paired_rows = []
for q in hot.index:
    h = hot.loc[q]
    m = mix.loc[q]
    identical = h["extracted_value"] == m["extracted_value"]
    if identical:
        pair_class = "identical_extracted_answer"
    elif m["exact_match"] and not h["exact_match"]:
        pair_class = "divergent_only_mixed_correct"
    elif h["exact_match"] and not m["exact_match"]:
        pair_class = "divergent_only_all_hot_correct"
    else:
        pair_class = "divergent_same_correctness"
    paired_rows.append(
        {
            "question_index": int(q),
            "session_id": h["session_id"],
            "batch": h["batch"],
            "gold_value": h["gold_value"],
            "all_hot_extracted": h["extracted_value"],
            "mixed_extracted": m["extracted_value"],
            "all_hot_exact_match": h["exact_match"],
            "mixed_exact_match": m["exact_match"],
            "all_hot_parse_failure": h["parse_failure"],
            "mixed_parse_failure": m["parse_failure"],
            "identical_extracted": identical,
            "pair_class": pair_class,
            "all_hot_generated_token_count": h["generated_token_count"],
            "mixed_generated_token_count": m["generated_token_count"],
            "mixed_read_steps": m["mixed_read_steps"],
            "warm_logical_blocks_observed": m["warm_logical_blocks_observed"],
            "all_hot_final_text": h["final_text"],
            "mixed_final_text": m["final_text"],
        }
    )
paired = pd.DataFrame(paired_rows).sort_values("question_index").reset_index(drop=True)

n = len(paired)
hot_exact = int(hot["exact_match"].sum())
mix_exact = int(mix["exact_match"].sum())
hot_parse = int(hot["parse_failure"].sum())
mix_parse = int(mix["parse_failure"].sum())
n_identical = int(paired["identical_extracted"].sum())
n_divergent = n - n_identical
only_hot = int((paired["all_hot_exact_match"] & ~paired["mixed_exact_match"]).sum())
only_mix = int((paired["mixed_exact_match"] & ~paired["all_hot_exact_match"]).sum())
both_correct = int((paired["all_hot_exact_match"] & paired["mixed_exact_match"]).sum())
both_wrong = int((~paired["all_hot_exact_match"] & ~paired["mixed_exact_match"]).sum())
same_corr_div = int((paired["pair_class"] == "divergent_same_correctness").sum())

print("\n=== GSM8K quality summary (n=20) ===")
print(f"All-HOT exact match: {hot_exact}/{n} ({100.0 * hot_exact / n:.0f}%)")
print(f"Mixed exact match: {mix_exact}/{n} ({100.0 * mix_exact / n:.0f}%)")
print(f"parse failures: {hot_parse} in All-HOT, {mix_parse} in Mixed")
print(f"identical extracted answers: {n_identical}/{n}")
print(f"divergent extracted answers: {n_divergent}/{n}")
print(f"only All-HOT correct: {only_hot}")
print(f"only Mixed correct: {only_mix}")
print(f"both correct: {both_correct}")
print(f"both wrong: {both_wrong}")
print(
    "n=20 is a small sample. The "
    f"{100.0 * mix_exact / n - 100.0 * hot_exact / n:.0f}-percentage-point "
    "difference is descriptive, not evidence that quantization improves accuracy."
)

assert (hot_exact, mix_exact) == (7, 8)
assert (hot_parse, mix_parse) == (1, 1)
assert (n_identical, n_divergent) == (17, 3)
assert (only_hot, only_mix, both_correct, both_wrong) == (0, 1, 7, 12)
assert same_corr_div == 2

summary_tbl = pd.DataFrame(
    [
        {"metric": "n_questions", "value": n},
        {"metric": "all_hot_exact_match", "value": hot_exact, "rate": hot_exact / n},
        {"metric": "mixed_exact_match", "value": mix_exact, "rate": mix_exact / n},
        {"metric": "all_hot_parse_failures", "value": hot_parse},
        {"metric": "mixed_parse_failures", "value": mix_parse},
        {
            "metric": "identical_extracted_answers",
            "value": n_identical,
            "rate": n_identical / n,
        },
        {
            "metric": "divergent_extracted_answers",
            "value": n_divergent,
            "rate": n_divergent / n,
        },
        {"metric": "only_all_hot_correct", "value": only_hot},
        {"metric": "only_mixed_correct", "value": only_mix},
        {"metric": "both_correct", "value": both_correct},
        {"metric": "both_wrong", "value": both_wrong},
        {"metric": "divergent_same_correctness_outcome", "value": same_corr_div},
        {"metric": "mixed_questions_eligible", "value": int(mix["eligible"].sum())},
        {"metric": "async_scheduling", "value": True},
        {"metric": "sequential_resume", "value": True},
        {
            "metric": "note",
            "value": (
                "n=20 is a small sample; the 5-percentage-point exact-match "
                "difference is descriptive, not evidence that quantization "
                "improves accuracy."
            ),
        },
    ]
)

per_question = sessions.copy()
divergent = paired.loc[~paired["identical_extracted"]].copy()

summary_csv = TABLES_DIR / "gsm8k_quality_summary.csv"
per_q_csv = TABLES_DIR / "gsm8k_per_question.csv"
div_csv = TABLES_DIR / "gsm8k_divergent_answers.csv"
summary_tbl.to_csv(summary_csv, index=False)
per_question.to_csv(per_q_csv, index=False)
divergent.to_csv(div_csv, index=False)
print("\nWrote", summary_csv.relative_to(REPO_ROOT))
print("Wrote", per_q_csv.relative_to(REPO_ROOT))
print("Wrote", div_csv.relative_to(REPO_ROOT))

print("\n=== Parse failures in either mode ===")
parse_fail_qs = paired[
    paired["all_hot_parse_failure"] | paired["mixed_parse_failure"]
]
if parse_fail_qs.empty:
    print("None")
else:
    for row in parse_fail_qs.itertuples():
        print(
            f"question_index={row.question_index} session_id={row.session_id} "
            f"gold={row.gold_value} "
            f"All-HOT parse_failure={row.all_hot_parse_failure} "
            f"extracted={row.all_hot_extracted} "
            f"Mixed parse_failure={row.mixed_parse_failure} "
            f"extracted={row.mixed_extracted}"
        )
        h = hot.loc[row.question_index]
        m = mix.loc[row.question_index]
        print("All-HOT final_text:\n", h["final_text"])
        print("Mixed final_text:\n", m["final_text"])
        print(
            "token counts All-HOT/Mixed:",
            int(h["generated_token_count"]),
            int(m["generated_token_count"]),
            "mixed_read_steps",
            int(m["mixed_read_steps"]),
            "warm_logical_blocks_observed",
            int(m["warm_logical_blocks_observed"]),
        )

print("\n=== Divergent extracted answers (3 questions) ===")
for row in divergent.itertuples():
    h = hot.loc[row.question_index]
    m = mix.loc[row.question_index]
    print("-" * 72)
    print(f"question_index={row.question_index} session_id={row.session_id}")
    print(f"gold_answer={row.gold_value}")
    print(
        f"All-HOT extracted={row.all_hot_extracted} exact_match={row.all_hot_exact_match} "
        f"parse_failure={row.all_hot_parse_failure} "
        f"generated_tokens={int(h['generated_token_count'])}"
    )
    print(
        f"Mixed extracted={row.mixed_extracted} exact_match={row.mixed_exact_match} "
        f"parse_failure={row.mixed_parse_failure} "
        f"generated_tokens={int(m['generated_token_count'])} "
        f"mixed_read_steps={int(m['mixed_read_steps'])} "
        f"warm_logical_blocks_observed={int(m['warm_logical_blocks_observed'])}"
    )
    print("All-HOT final generated text:")
    print(h["final_text"])
    print("Mixed final generated text:")
    print(m["final_text"])

pair_order = [
    ("identical_extracted_answer", "Identical extracted answer"),
    ("divergent_same_correctness", "Divergent, same correctness outcome"),
    ("divergent_only_mixed_correct", "Divergent, only Mixed correct"),
    ("divergent_only_all_hot_correct", "Divergent, only All-HOT correct"),
]
pair_counts = paired["pair_class"].value_counts()
counts = [int(pair_counts.get(key, 0)) for key, _ in pair_order]
labels = [label for _, label in pair_order]
assert counts == [17, 2, 1, 0], counts

fig, ax = plt.subplots(figsize=(7.2, 3.6))
bar_colors = [COLOR_ALL_HOT, "#6b7c93", COLOR_MIXED, "#9aa5b1"]
ys = list(range(len(labels)))[::-1]
ax.barh(ys, counts[::-1], color=bar_colors[::-1], height=0.62, zorder=3)
for y, count in zip(ys, counts[::-1]):
    ax.text(count + 0.15, y, str(count), va="center", ha="left", fontsize=11)
ax.set_yticks(ys)
ax.set_yticklabels(labels[::-1])
ax.set_xlabel("Questions (count)")
ax.set_xlim(0, 20)
ax.set_title("Paired GSM8K resume-quality outcomes (n = 20 questions)")
ax.set_xticks(range(0, 21, 2))
fig.text(
    0.0,
    -0.08,
    "Counts only. n = 20 is a small sample; the 5-percentage-point exact-match "
    "difference is descriptive, not evidence that quantization improves accuracy.\n"
    "Source: gsm8k_resume_quality/final_n20_batches5_cap512_seq, four batch_* directories.",
    ha="left",
    va="top",
    fontsize=9,
    color="0.25",
)
save_figure(fig, "gsm8k_resume_quality")
plt.show()


GSM8K root: experiments/hkv_pressure_study/runs/gsm8k_resume_quality/final_n20_batches5_cap512_seq
batch directories: ['batch_1_start_0', 'batch_2_start_5', 'batch_3_start_10', 'batch_4_start_15']
JSON files:
  experiments/hkv_pressure_study/runs/gsm8k_resume_quality/final_n20_batches5_cap512_seq/batch_1_start_0/all_hot.json
  experiments/hkv_pressure_study/runs/gsm8k_resume_quality/final_n20_batches5_cap512_seq/batch_1_start_0/mixed.json
  experiments/hkv_pressure_study/runs/gsm8k_resume_quality/final_n20_batches5_cap512_seq/batch_2_start_5/all_hot.json
  experiments/hkv_pressure_study/runs/gsm8k_resume_quality/final_n20_batches5_cap512_seq/batch_2_start_5/mixed.json
  experiments/hkv_pressure_study/runs/gsm8k_resume_quality/final_n20_batches5_cap512_seq/batch_3_start_10/all_hot.json
  experiments/hkv_pressure_study/runs/gsm8k_resume_quality/final_n20_batches5_cap512_seq/batch_3_start_10/mixed.json
  experiments/hkv_pressure_study/runs/gsm8k_resume_quality/final_n20_batches5_cap512_se

,batch,mode,source_path,async_scheduling,sequential_resume,validation_passed,n_selected,n_eligible,n_exact_match,n_parse_failures,warm_observed,peak_warm_blocks,allocator_consistent,cleanup_complete
0,batch_1_start_0,all-hot,experiments/hkv_pressure_study/runs/gsm8k_resume_quality/final_n20_batches5_...,True,True,True,5,5,2,0,False,0,True,True
1,batch_1_start_0,mixed,experiments/hkv_pressure_study/runs/gsm8k_resume_quality/final_n20_batches5_...,True,True,True,5,5,2,0,True,357,True,True
2,batch_2_start_5,all-hot,experiments/hkv_pressure_study/runs/gsm8k_resume_quality/final_n20_batches5_...,True,True,True,5,5,1,1,False,0,True,True
3,batch_2_start_5,mixed,experiments/hkv_pressure_study/runs/gsm8k_resume_quality/final_n20_batches5_...,True,True,True,5,5,1,1,True,358,True,True
4,batch_3_start_10,all-hot,experiments/hkv_pressure_study/runs/gsm8k_resume_quality/final_n20_batches5_...,True,True,True,5,5,3,0,False,0,True,True
5,batch_3_start_10,mixed,experiments/hkv_pressure_study/runs/gsm8k_resume_quality/final_n20_batches5_...,True,True,True,5,5,3,0,True,358,True,True
6,batch_4_start_15,all-hot,experiments/hkv_pressure_study/runs/gsm8k_resume_quality/final_n20_batches5_...,True,True,True,5,5,1,0,False,0,True,True
7,batch_4_start_15,mixed,experiments/hkv_pressure_study/runs/gsm8k_resume_quality/final_n20_batches5_...,True,True,True,5,5,2,0,True,358,True,True


async_scheduling=true for every file
sequential_resume=true for every file
validation passed for every file
all 20 Mixed questions eligible
WARM observed in every Mixed batch
allocator consistency and cleanup passed everywhere

=== GSM8K quality summary (n=20) ===
All-HOT exact match: 7/20 (35%)
Mixed exact match: 8/20 (40%)
parse failures: 1 in All-HOT, 1 in Mixed
identical extracted answers: 17/20
divergent extracted answers: 3/20
only All-HOT correct: 0
only Mixed correct: 1
both correct: 7
both wrong: 12
n=20 is a small sample. The 5-percentage-point difference is descriptive, not evidence that quantization improves accuracy.

Wrote experiments/hkv_pressure_study/tables/gsm8k_quality_summary.csv
Wrote experiments/hkv_pressure_study/tables/gsm8k_per_question.csv
Wrote experiments/hkv_pressure_study/tables/gsm8k_divergent_answers.csv

=== Parse failures in either mode ===
question_index=5 session_id=gsm8k-0005 gold=64 All-HOT parse_failure=True extracted=-9999999 Mixed parse_failure=

## 9. Export inventory


In [12]:
csv_files = sorted(TABLES_DIR.glob("*.csv"))
plot_files = sorted([p for p in PLOTS_DIR.iterdir() if p.suffix in {".pdf", ".png"}])
print("CSV tables:")
for p in csv_files:
    print(" ", p.relative_to(REPO_ROOT))
print("\nFigures:")
for p in plot_files:
    print(" ", p.relative_to(REPO_ROOT))


CSV tables:
  experiments/hkv_pressure_study/tables/all_runs_summary.csv
  experiments/hkv_pressure_study/tables/boundary_125_aggregate.csv
  experiments/hkv_pressure_study/tables/boundary_125_repetitions.csv
  experiments/hkv_pressure_study/tables/full_trace_overload.csv
  experiments/hkv_pressure_study/tables/gsm8k_divergent_answers.csv
  experiments/hkv_pressure_study/tables/gsm8k_per_question.csv
  experiments/hkv_pressure_study/tables/gsm8k_quality_summary.csv
  experiments/hkv_pressure_study/tables/memory_composition.csv

Figures:
  experiments/hkv_pressure_study/plots/boundary_125_completed_requests.pdf
  experiments/hkv_pressure_study/plots/boundary_125_completed_requests.png
  experiments/hkv_pressure_study/plots/boundary_125_completed_sessions.pdf
  experiments/hkv_pressure_study/plots/boundary_125_completed_sessions.png
  experiments/hkv_pressure_study/plots/boundary_125_time_to_last_completed_turn.pdf
  experiments/hkv_pressure_study/plots/boundary_125_time_to_last_complete

## 10. Paper-ready findings

Values in this cell are computed from the loaded JSON files only. Timed-out runs
are not described as completed. Performance-mode `validation.passed` records
operational checks (timeout, allocator consistency, cleanup); it does **not**
establish output-quality equivalence between All-HOT and Mixed.


In [13]:
def fmt_num(x, digits=2, suffix=""):
    if x is None or (isinstance(x, float) and (math.isnan(x) or math.isinf(x))):
        return "n/a"
    if isinstance(x, (int, np.integer)):
        return f"{int(x):,}{suffix}"
    return f"{float(x):.{digits}f}{suffix}"


def mode_slice(df, mode):
    return df[df["mode"] == mode]


print("=== Loaded corpus ===")
print(f"Result files discovered: {len(discovered_paths)}")
n_flagged = int(diag_df['issue'].notna().sum()) if not diag_df.empty else 0
print(f"Files flagged/skipped: {n_flagged}")
if not summary.empty:
    print(
        "Completed runs: "
        f"{int(summary['completed_run'].sum())} / {len(summary)}"
    )
    print(
        "Timed-out runs: "
        f"{int(summary['timed_out'].fillna(False).sum())} / {len(summary)}"
    )

print("\n=== Boundary completion (selected-request %) ===")
if boundary.empty:
    print("No boundary runs loaded.")
else:
    for size in sorted(boundary["workload_size"].dropna().unique()):
        block = boundary[boundary["workload_size"] == size]
        for mode in ("all-hot", "mixed"):
            sub = mode_slice(block, mode)
            if sub.empty:
                continue
            rates = sub["completion_rate_pct"]
            n_to = int(sub["timed_out"].fillna(False).sum())
            n_ok = int(sub["completed_run"].sum())
            print(
                f"  {int(size)} sessions {MODE_LABELS[mode]}: "
                f"mean completion {fmt_num(rates.mean(), 1)}% "
                f"(min {fmt_num(rates.min(), 1)}, max {fmt_num(rates.max(), 1)}; "
                f"n={len(sub)}, completed={n_ok}, timed_out={n_to})"
            )

print("\n=== 125-session repetitions ===")
if rep125.empty:
    print("No 125-session repetitions loaded.")
else:
    for mode in ("all-hot", "mixed"):
        sub = mode_slice(rep125, mode)
        if sub.empty:
            continue
        print(f"  {MODE_LABELS[mode]}:")
        print(
            "    completed requests: "
            + ", ".join(
                f"r{int(r.repetition)}={int(r.completed_requests)}/"
                f"{int(r.selected_requests)}"
                f"{'' if r.completed_run else ' (timeout)'}"
                for r in sub.sort_values("repetition").itertuples()
            )
        )
        print(
            "    completed sessions: "
            + ", ".join(
                f"r{int(r.repetition)}={int(r.completed_sessions)}/"
                f"{int(r.selected_sessions)}"
                f"{'' if r.completed_run else ' (timeout)'}"
                for r in sub.sort_values("repetition").itertuples()
            )
        )
        tlt = sub["time_to_last_completed_turn_s"]
        print(
            "    time to last completed turn (s): "
            f"mean {fmt_num(tlt.mean())}, min {fmt_num(tlt.min())}, max {fmt_num(tlt.max())}"
        )
        done = sub[sub["completed_run"]]
        timed = sub[~sub["completed_run"]]
        if not done.empty and "requests_per_second_completed_workload" in done:
            rps = done["requests_per_second_completed_workload"]
            tps = done["output_tokens_per_second_completed_workload"]
            print(
                "    completed-workload throughput: "
                f"requests/s mean {fmt_num(rps.mean(), 3)} "
                f"(n={len(done)}); "
                f"output tokens/s mean {fmt_num(tps.mean(), 2)}"
            )
        if not timed.empty:
            print(
                f"    timed-out repetitions excluded from throughput/latency averages: {len(timed)}"
            )

print("\n=== Persistent KV budget ===")
if memory_df.empty:
    print("No memory records.")
else:
    for mode in ("all-hot", "mixed"):
        sub = mode_slice(memory_df, mode)
        if sub.empty:
            continue
        row = sub.iloc[0]
        print(
            f"  {MODE_LABELS[mode]}: HOT {fmt_num(giB(row['hot_bytes']), 3)} GiB, "
            f"WARM {fmt_num(giB(row['warm_bytes']), 3)} GiB, "
            f"metadata {fmt_num(giB(row['metadata_bytes']), 4)} GiB, "
            f"total {fmt_num(giB(row['total_persistent_kv_bytes']), 3)} GiB, "
            f"configured {fmt_num(giB(row['configured_kv_budget_bytes']), 3)} GiB, "
            f"slack {int(row['budget_slack_bytes']) if pd.notna(row['budget_slack_bytes']) else 'n/a'} B"
        )
    print(
        f"  Slack identity holds for {int(memory_df['within_recorded_slack'].sum())}/"
        f"{len(memory_df)} loaded runs."
    )

print("\n=== Workload classification ===")
if not summary.empty:
    print(summary.groupby(["workload_family", "run_dir_name", "mode_label"]).size().rename("n").to_string())

print("\n=== Full-trace overload ===")
if overload.empty:
    print("No full-trace overload files loaded.")
else:
    for r in overload.sort_values("mode").itertuples():
        sess_pct = (
            100.0 * float(r.completed_sessions) / float(r.selected_sessions)
            if r.selected_sessions
            else float("nan")
        )
        print(
            f"  {r.mode_label} {r.run_label}: "
            f"sessions {int(r.completed_sessions)}/{int(r.selected_sessions)} "
            f"({fmt_num(sess_pct, 2)}%), "
            f"requests {int(r.completed_requests)}/{int(r.selected_requests)} "
            f"({fmt_num(r.completion_rate_pct, 2)}%), "
            f"generated tokens {fmt_num(r.generated_tokens, 0)}, "
            f"service {fmt_num(r.service_window_duration_s)} s, "
            f"seconds since last turn {fmt_num(r.seconds_since_last_completed_turn)}, "
            f"time of last completed turn {fmt_num(r.time_to_last_completed_turn_s)} s, "
            f"timed_out={bool(r.timed_out)}"
        )

print("\n=== Nominal load ===")
if nominal.empty:
    print(
        "Nominal-load data are unavailable in the currently discovered run directory."
    )
elif nominal["completed_run"].any():
    done = nominal[nominal["completed_run"]]
    for r in done.itertuples():
        print(
            f"  {r.mode_label} {r.run_label}: "
            f"{fmt_num(r.requests_per_second_completed_workload, 3)} req/s, "
            f"TTFT p50={fmt_num(r.ttft_p50_s, 3)} s, "
            f"resumed TTFT p50={fmt_num(r.resumed_ttft_p50_s, 3)} s, "
            f"latency p50={fmt_num(r.latency_p50_s, 3)} s, "
            f"resumed latency p50={fmt_num(r.resumed_latency_p50_s, 3)} s"
        )
else:
    print(
        "Nominal-load files were found but none completed; "
        "throughput and latency are not treated as complete-workload results."
    )

print("\n=== Validation caveat ===")
if not summary.empty:
    n_perf = int((summary["experiment_mode"] == "performance").sum())
    print(
        f"{n_perf}/{len(summary)} parsed files record experiment_mode='performance'. "
        "Do not interpret validation.passed or token diagnostics as output-quality equivalence."
    )
print("\n=== GSM8K resume-quality ===")
if "hot" in globals() and "mix" in globals():
    print(
        f"All-HOT exact match {int(hot['exact_match'].sum())}/20; "
        f"Mixed exact match {int(mix['exact_match'].sum())}/20. "
        "n=20 is a small sample; the 5-percentage-point difference is descriptive, "
        "not evidence that quantization improves accuracy."
    )
else:
    print("GSM8K section did not produce summary tables.")


=== Loaded corpus ===
Result files discovered: 12
Files flagged/skipped: 0
Completed runs: 5 / 12
Timed-out runs: 7 / 12

=== Boundary completion (selected-request %) ===
  100 sessions All-HOT: mean completion 100.0% (min 100.0, max 100.0; n=1, completed=1, timed_out=0)
  100 sessions Mixed: mean completion 100.0% (min 100.0, max 100.0; n=1, completed=1, timed_out=0)
  125 sessions All-HOT: mean completion 55.6% (min 53.5, max 57.5; n=3, completed=0, timed_out=3)
  125 sessions Mixed: mean completion 100.0% (min 100.0, max 100.0; n=3, completed=3, timed_out=0)
  150 sessions All-HOT: mean completion 47.9% (min 47.9, max 47.9; n=1, completed=0, timed_out=1)
  150 sessions Mixed: mean completion 57.6% (min 57.6, max 57.6; n=1, completed=0, timed_out=1)

=== 125-session repetitions ===
  All-HOT:
    completed requests: r1=158/275 (timeout), r2=154/275 (timeout), r3=147/275 (timeout)
    completed sessions: r1=70/125 (timeout), r2=69/125 (timeout), r3=67/125 (timeout)
    time to last co